# Reading JSON Data

## Creating a DataFrame using JSON Data

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
import json

spark = SparkSession.builder.appName("JsonExample").getOrCreate()

# Sample data with JSON strings
data = [("1", '{"name": "John", "age": 30}'), ("2", '{"name": "Alice", "age": 25}')]

# Creating DataFrame
df_json1 = spark.createDataFrame(data, ["id", "json_data"])

df_json1.show()

## Reading JSON data from a file and create a Dataframe out of it

In [0]:
df_json2 = spark.read.format('json').option('multiline','true').load('/Workspace/Users/sandipan.kar.data@gmail.com/Practice Dataset/JSON_Data_21.csv')
display(df_json2)

In [0]:
df_json2.printSchema()

# Different Schema Functions

## from_json

* Parses a JSON string column into a StructType.

In [0]:
display(df_json1)

In [0]:
from pyspark.sql.functions import col
display(df_json1.withColumn('schema_json_data', schema_of_json(col('json_data'))))

In [0]:
from pyspark.sql.functions import from_json
from pyspark.sql.types import *

schema = StructType([
    StructField("name", StringType()),
    StructField("age", IntegerType()),
])

df_schemaed = df_json1.withColumn(
    "parsed_json",
    from_json("json_data", schema)
)
display(df_schemaed)

## to_json
* Converts a column containing complex data types like StructType, ArrayType, MapType, or VariantType into a JSON-formatted string column.

In [0]:
from pyspark.sql.functions import to_json
from pyspark.sql.types import *

df_json_text = df_schemaed.withColumn(
    "json_string",
    to_json("parsed_json")
)
display(df_json_text)

## get_json_object
* Extract single field

In [0]:
from pyspark.sql.functions import get_json_object

df_json1.select(
    get_json_object("json_data", "$.name")
).show()

df_schemaed.select(
    get_json_object("json_data", "$.name")
).show()

## json_tuple
* Extract multiple fields

In [0]:
from pyspark.sql.functions import json_tuple

df_json1.select(
    json_tuple("json_data", "age", "name")
).withColumnsRenamed(
    {"c0": "age", "c1": "name"}
).show()

## schema_of_json
* Infer JSON Schema

In [0]:
from pyspark.sql.functions import schema_of_json, col

display(
    df_json1.select(
    schema_of_json(
        col('json_data')
    )
))

## Parsing Complex JSON data using explode
* Flatten JSON array

In [0]:
df_json2.printSchema()

In [0]:
from pyspark.sql.functions import explode
df_json2_norm = df_json2.withColumn('batter_normalize', explode(col('batters.batter'))).withColumn('topping_normalize', explode(col('topping'))).drop('batters', 'topping')
df_json2_norm = df_json2_norm.select('id', 'name', 'ppu', 'type', col('batter_normalize.id').alias('batter_id'), col('batter_normalize.type').alias('batter_type'), col('topping_normalize.id').alias('topping_id'), col('topping_normalize.type').alias('topping_type'))
display(df_json2_norm)